# Chapter 1 - Introduction to Language Models
*Exploring the exciting field of Language AI*

『핸즈온 LLM』 1장 실습 노트북

> 출처: [HandsOnLLM/Hands-On-Large-Language-Models](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models/tree/main/chapter01) (Apache-2.0)  
> 공식 노트북을 가져와 주석을 덧붙였습니다.

### [OPTIONAL] - Installing Packages on Colab

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** the following codeblock to install the dependencies for this chapter:

---

💡 **NOTE**: We will want to use a GPU to run the examples in this notebook. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

---

In [ ]:
# Colab에서 돌릴 때만 주석 풀고 실행
# %%capture
# !pip install transformers==4.41.2 accelerate==0.31.0

# Phi-3

The first step is to load our model onto the GPU for faster inference. Note that we load the model and tokenizer separately (although that isn't always necessary).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Phi-3 mini 불러오기. GPU(T4)에 올려야 생성 속도가 버틸 만함
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False,
)
# 토크나이저도 같은 모델 이름으로 따로 불러옴
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

Although we can now use the model and tokenizer directly, it's much easier to wrap it in a `pipeline` object:

In [ ]:
from transformers import pipeline

# 모델이랑 토크나이저를 pipeline으로 묶어두면 프롬프트 넣고 답 받는 게 한 줄로 끝남
# return_full_text=False: 내가 넣은 프롬프트는 빼고 새로 생성된 부분만 받기
# do_sample=False: 샘플링을 끄면 매번 같은 답(greedy)이 나와서 결과 비교하기 편함
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=500,
    do_sample=False
)

Finally, we create our prompt as a user and give it to the model:

In [ ]:
# 채팅 모델이라 role/content 형식의 메시지 리스트로 넣어줌
messages = [
    {"role": "user", "content": "Create a funny joke about chickens."}
]

# 결과가 [{"generated_text": ...}] 모양이라 텍스트만 꺼내서 출력
output = generator(messages)
print(output[0]["generated_text"])